# 01 — EDA i kontrakt danych: Iranian Churn (Sprint 1)

Analiza pełnego zbioru UCI: przewidywanie odejścia klienta na koniec 12. miesiąca
na podstawie informacji z miesięcy 1–9. Target: `Churn`, klasa pozytywna: `1`.
Użytkownik: dział zatrzymania klienta, planujący działania po 9. miesiącu.

Uruchomienie: środowisko `asi-ml`, pakiet projektu zainstalowany zgodnie z README,
`python scripts/download_data.py`, następnie Restart & Run All.
Domyślnie analizujemy **pełne dane**, nie próbkę CI. `ASI_DATA_MODE=sample` służy
wyłącznie sprawdzeniu działania; poniższe wnioski liczbowe odnoszą się do pełnych danych.
Notebook w Git pozostaje bez outputów zgodnie z zasadami repozytorium.

Źródło: [UCI Iranian Churn](https://archive.ics.uci.edu/dataset/563/iranian+churn+dataset),
[DOI](https://doi.org/10.24432/C5JW3Z), CC BY 4.0.
Kontrakt prowadzącej (`src/asi_project/datasets.yaml`) pozostaje bez zmian.

## 0. Ustawienia

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from asi_project.config import get_contract, load_project_config
from asi_project.data import (
    assert_valid,
    encode_target,
    feature_list_problems,
    filter_training_rows,
    is_text_column,
    load_dataset,
    normalize_input,
    target_classes,
    validate,
)

config = load_project_config()
contract = get_contract(config)
DATA_MODE = os.environ.get("ASI_DATA_MODE", "full")

print(f"Dataset: {contract.title} ({contract.dataset_id})")
print(f"Target: {contract.target} | klasa pozytywna: {contract.positive_class}")
print(f"Moment predykcji: {contract.prediction_moment}")

## 1. Wczytanie danych

Pełne dane pobiera `python scripts/download_data.py`. Nazwy kolumn są ujednolicane przy
wczytywaniu (białe znaki), wartości — jeszcze nie.

In [ ]:
df_raw = load_dataset(config, contract, mode=DATA_MODE)
print(df_raw.shape)
df_raw.head()

## 2. Inspekcja: typy, braki, liczba unikalnych wartości

In [ ]:
overview = pd.DataFrame(
    {
        "dtype": df_raw.dtypes.astype(str),
        "braki": df_raw.isna().sum(),
        "unikalne": df_raw.nunique(),
    },
)
overview

**Obserwacje i decyzje.** Pełny zbiór zawiera 3150 rekordów i 14 kolumn,
w tym target oraz wykluczoną cechę `Customer Value`. Nie ma braków rozpoznanych przez pandas;
wszystkie kolumny są liczbowe, ale `Complains`, `Charge Amount`, `Age Group`, `Tariff Plan`
i `Status` opisują kategorie. Pozostawiamy ich deklarację jako kategorycznych w `project.yaml`.
`Age` ma tylko pięć wartości i dokładnie odpowiada `Age Group`; zachowujemy obie cechy
w pierwszym baseline zgodnie z konfiguracją zespołu, ale dokumentujemy redundancję.

## 3. Kontrakt danych

Kontrakt prowadzącej określa target, klasę pozytywną, ALLOWED FEATURES (jedyne kolumny
dozwolone w X) i EXCLUDED FEATURES wraz z powodem wykluczenia.

In [ ]:
assert_valid(df_raw, contract)
problems = validate(df_raw, contract)
print("Kontrakt danych:", "OK" if not problems else problems)
print("ALLOWED FEATURES:", list(contract.allowed_features))
print("EXCLUDED FEATURES:")
for column, reason in contract.excluded_features.items():
    status = "jest w danych" if column in df_raw.columns else "brak w pliku"
    print(f"- {column} ({status}): {reason}")

## 4. Target i klasa pozytywna

In [ ]:
print("Klasy targetu:", target_classes(df_raw[contract.target]))
y = encode_target(df_raw[contract.target].dropna(), contract.positive_class)
print(f"Udział klasy pozytywnej: {y.mean():.3f}")
ax = y.value_counts().sort_index().plot(kind="bar", title="Target (1 = klasa pozytywna)")
ax.set_xlabel("klasa")
plt.show()

**Metryka główna: F1 dla klasy `Churn=1`.** W pełnym zbiorze 495 z 3150 klientów
odeszło (15,71%), a 2655 pozostało. Przewidywanie wyłącznie braku odejścia daje już
84,29% accuracy, choć nie wykrywa żadnego odchodzącego klienta. F1 uwzględnia zarówno
wykrywanie odejść (recall), jak i trafność kierowania ofert (precision), co odpowiada
uzasadnieniu w karcie projektu. Koszty błędów nie zostały wycenione, więc nie twierdzimy,
że F1 optymalizuje wynik finansowy. Precision, recall i macierz pomyłek ocenimy w baseline.

## 5. EDA: dozwolone cechy i zależności z targetem

Analizy dotyczą wyłącznie ALLOWED FEATURES. Target służy do porównywania grup;
nie jest cechą wejściową. `Customer Value` nie bierze udziału w analizach predyktorów.
Zależności opisują ten zbiór, nie dowodzą przyczyn odejścia ani skuteczności rabatów.

In [ ]:
numeric = list(config.numeric_features)
display(df_raw.groupby(contract.target)[numeric].agg(["median", "mean"]).round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
bins = np.histogram_bin_edges(df_raw["Seconds of Use"], bins=30)
for label, group in df_raw.groupby(contract.target):
    axes[0].hist(
        group["Seconds of Use"], bins=bins, density=True, alpha=0.5, label=f"Churn={label}"
    )
axes[0].set(title="Czas rozmów według klasy", xlabel="Seconds of Use", ylabel="Gęstość")
axes[0].legend()
df_raw.boxplot(column="Frequency of SMS", by=contract.target, ax=axes[1])
axes[1].set(title="Liczba SMS według klasy", xlabel="Churn", ylabel="Frequency of SMS")
fig.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for column in config.categorical_features:
    table = df_raw.groupby(column)[contract.target].agg(liczba="size", odejscia="sum")
    table["odsetek_odejsc_pct"] = (100 * table["odejscia"] / table["liczba"]).round(2)
    print(column)
    display(table)
    if column in ("Complains", "Status"):
        ax = axes[0 if column == "Complains" else 1]
        table["odsetek_odejsc_pct"].plot.bar(ax=ax, rot=0)
        ax.set(title=f"Odejścia a {column}", xlabel=column, ylabel="Odsetek odejść [%]")
        ax.axhline(100 * y.mean(), color="gray", linestyle="--", label="Cały zbiór")
        ax.legend()
        ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

**Wnioski z pełnych danych.** Klienci, którzy odeszli, mają niższą medianę czasu rozmów
(1182 wobec 3530 sekund), liczby połączeń (25 wobec 63) i SMS-ów (9 wobec 25).
Wśród zgłaszających reklamację odchodzi 200 z 241 klientów (82,99%), a wśród pozostałych
295 z 2909 (10,14%). Dla `Status=2` udział odejść to 47,31%, a dla `Status=1` — 5,28%;
cecha jest dozwolona, ponieważ zgodnie z kontraktem opisuje okres miesięcy 1–9.
Grupy rzadkie, np. `Tariff Plan=2` z sześcioma odejściami na 245 klientów, wymagają
ostrożnej interpretacji. Zachowujemy dozwolone cechy w baseline; silna zależność
nie jest sama w sobie dowodem leakage, a ocenę skuteczności wykonamy na odłożonym teście.

### 5a. Zakresy, zera i wartości specjalne

Sprawdzamy wszystkie dozwolone cechy. Wartości rzadkie i skrajne wymagają interpretacji,
nie automatycznego usuwania. Zestawienie kategorii pokazuje też rozbieżności z opisem danych.

In [ ]:
display(df_raw[list(contract.allowed_features)].describe().T)
print("Kody kategorii:")
for column in config.categorical_features:
    print(column, sorted(df_raw[column].dropna().unique().tolist()))
zeros = df_raw.groupby(df_raw["Seconds of Use"].eq(0))[contract.target].agg(
    liczba="size", odejscia="sum", udzial="mean"
)
print("Czy Seconds of Use = 0:")
display(zeros)
print("Ujemne wartości cech numerycznych:")
display(df_raw[numeric].lt(0).sum())

**Obserwacja → decyzja.** Nie ma ujemnych wartości cech numerycznych. Czas rozmów
wynosi 0–17090 sekund, a liczba SMS 0–522; duże wartości nie są wystarczającym powodem
do usuwania klientów. Zerowy czas rozmów ma 154 klientów, z których 81 odchodzi (52,60%);
nie traktujemy zera jako braku. `Charge Amount` ma kody **0–10**, mimo zakresu 0–9
w `docs/datasets.md`; kod 10 występuje w siedmiu rekordach. Zachowujemy go jako kategorię
i odnotowujemy rozbieżność, bez zmieniania kontraktu i bez uznawania rekordu za błędny.

### 5b. Redundancja cech i zgodność etykiet

In [ ]:
print("Wiek według Age Group:")
display(pd.crosstab(df_raw["Age Group"], df_raw["Age"]))
corr = df_raw[numeric].corr()
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(numeric)), numeric, rotation=60, ha="right")
ax.set_yticks(range(len(numeric)), numeric)
for row in range(len(numeric)):
    for col in range(len(numeric)):
        ax.text(col, row, f"{corr.iloc[row, col]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, label="Korelacja Pearsona")
ax.set_title("Korelacje dozwolonych cech numerycznych")
plt.tight_layout()
plt.show()

profiles = df_raw.groupby(list(contract.allowed_features), dropna=False)[contract.target]
print(
    "Nadmiarowe wiersze o identycznych cechach:",
    df_raw.duplicated(subset=list(contract.allowed_features)).sum(),
)
print("Profile z różnymi etykietami:", profiles.nunique().gt(1).sum())
print("Liczebności i udział odejść po usunięciu dokładnych duplikatów (diagnostyka):")
display(df_raw.drop_duplicates()[contract.target].agg(["count", "sum", "mean"]))

**Obserwacja → decyzja.** Grupy wieku 1–5 odpowiadają dokładnie wartościom
15, 25, 30, 45 i 55 lat. Korelacja czasu i liczby połączeń wynosi około 0,95.
Pozostawiamy te cechy w pierwszym baseline, wskazując redundancję jako temat do kolejnych
eksperymentów, a nie podstawę automatycznej selekcji. Jest 314 nadmiarowych wierszy według
samych dozwolonych cech oraz 14 profili z różnymi etykietami. Diagnostyczne usunięcie
dokładnych duplikatów pozostawia 2850 rekordów i 446 odejść (15,65%); pokazuje to skalę
problemu, ale nie rozstrzyga, czy klienci są tymi samymi osobami.

## 6. Input normalization

Input normalization działa na pojedynczym rekordzie, nie używa targetu ani statystyk zbioru
i jest idempotentna. Typowe kandydatki: białe znaki, puste napisy, oczywiste reprezentacje
braków, ujednolicenie zapisu wartości. Uzupełnianie braków medianą **nie jest** input
normalization — wymaga `fit`, więc należy do model preprocessing.

In [ ]:
text_columns = [column for column in df_raw.columns if is_text_column(df_raw[column])]
for column in text_columns:
    values = df_raw[column].dropna().astype(str)
    padded = int((values != values.str.strip()).sum())
    empty = int((values.str.strip() == "").sum())
    top = values.value_counts().head(6).to_dict()
    print(f"{column}: spacje na brzegach={padded}, puste={empty}, najczęstsze={top}")

**Decyzja o input normalization.** Zachowujemy istniejące przycinanie spacji i zamianę
pustych napisów na brak. Nie znaleziono podstaw do dodawania innych reguł dla tego pliku:
zera oznaczają możliwy brak aktywności, a nie potwierdzony kod brakującej wartości.
Nie zamieniamy ich na NaN, nie obcinamy dużych wartości i nie zmieniamy kodów kategorii.
Reguły obecnej funkcji działają na jednym rekordzie, bez targetu i statystyk zbioru,
a ponowne wywołanie nie zmienia wyniku. Imputacja, skalowanie i kodowanie kategorii
pozostają zadaniem pipeline'u modelu, dopasowywanego wyłącznie na treningu.

In [ ]:
normalized = normalize_input(df_raw)
changed = (normalized.astype(str) != df_raw.astype(str)).sum()
changed[changed > 0]

## 7. Training-only filtering

Training-only filtering działa tylko przy przygotowaniu danych treningowych i może jedynie
usuwać rekordy. Domyślnie `filter_training_rows()` usuwa rekordy bez targetu.

In [ ]:
print("Rekordy bez targetu:", int(df_raw[contract.target].isna().sum()))
print("Dokładne duplikaty wierszy:", int(df_raw.duplicated().sum()))
filtered = filter_training_rows(normalize_input(df_raw), contract)
print(f"Po training-only filtering: {len(filtered)} z {len(df_raw)} rekordów")

**Decyzja o filtrowaniu.** Zachowujemy istniejące usuwanie rekordów bez targetu;
w pełnym zbiorze nie usuwa to żadnego wiersza. Występuje 300 dokładnych duplikatów
wierszy (9,52%), jednak kontrakt opisuje rekordy jako różnych klientów i nie ma ich
identyfikatorów. Nie da się potwierdzić, że identyczne agregaty są powtórzeniem tej samej
osoby, dlatego nie usuwamy ich automatycznie. Jest też 14 grup identycznych ALLOWED FEATURES
z różnymi etykietami; nie poprawiamy ani nie odrzucamy tych etykiet arbitralnie.

**Ograniczenie oceny:** identyczne profile mogą znaleźć się po obu stronach wymaganego
splitu `stratified_random`, co może zawyżać ocenę generalizacji na nowe profile.
Dokumentujemy to ryzyko; sam zbiór nie pozwala dowieść, że ta sama osoba jest w train i test.
Nie zmieniamy strategii prowadzącej. Dodatkowa analiza wrażliwości na duplikaty może być
osobnym eksperymentem, bez dobierania reguł pod wynik testowy.

## 8. Typy cech → `project.yaml`

Potwierdzamy obecny podział: siedem cech numerycznych i pięć kategorycznych.
`Complains`, `Tariff Plan` i `Status` są kodami kategorii; `Charge Amount` i `Age Group`
traktujemy kategorycznie, aby baseline nie narzucał liniowej zależności od kodu.
`Age` pozostaje numeryczne jako wiek w latach, mimo pięciu wartości; `Age Group` jest
redundantne i warto później porównać wariant z jedną reprezentacją wieku na treningu.
Konfiguracja nie wymaga zmiany. Target i `Customer Value` pozostają poza X.

In [ ]:
assert_valid(normalized, contract, config)
print("Numeryczne:", list(config.numeric_features))
print("Kategoryczne:", list(config.categorical_features))
print("Problemy z deklaracją:", feature_list_problems(contract, config.model_features) or "brak")
print("Kontrakt z deklaracją:", validate(normalize_input(df_raw), contract, config) or "OK")

## 9. Podsumowanie decyzji

- **Wejście:** bez dodatkowych reguł; zachowujemy przycinanie spacji i puste napisy → NaN.
  Zera aktywności i kod `Charge Amount=10` pozostają bez zmian.
- **Filtrowanie:** tylko rekordy bez targetu, tak jak w obecnym kodzie. Nie usuwamy
  automatycznie duplikatów, zer aktywności ani obserwacji odstających; ryzyko identycznych
  profili w train/test jest opisanym ograniczeniem.
- **Cechy:** potwierdzamy 7 numerycznych i 5 kategorycznych z `project.yaml`.
  Dokumentujemy redundancję wieku oraz silną korelację miar aktywności. `Customer Value`
  i target nie są predyktorami; kontrakt prowadzącej pozostaje bez zmian.
- **Ocena:** główna metryka to F1 dla odejść, uzupełniona precision, recall i macierzą pomyłek.
  Split zgodny z kontraktem: `stratified_random`, test 20%, seed 42.
- **Kolejny etap:** implementacja imputacji, skalowania i kodowania w sklearn Pipeline,
  porównanie DummyClassifier z LogisticRegression w notebooku 02. Te operacje wymagają
  dopasowania na treningu; EDA nie zastępuje baseline ani nie dostraja modelu na teście.

Wyniki liczbowe dotyczą pełnego pliku o SHA-256
`90d5fb6bd1630cd4de4b4d28fcf8b4cb92a8f6ab7484605b0799d47386f7dbe1`.
Decyzje stanowią materiał do ADR-1 w Sprincie 2.